In [ ]:
import marimo as mo

# Higher-Order Models for Time-Respecting Paths in Temporal Graphs

*August 4 2026*
*Training Workshop: Causality-Aware Temporal Networks*
*Ingo Scholtes, CAIDAS, Julius-Maximilians-Universität Würzburg (JMU), Germany*

## Motivation

In the previous tutorial, we have seen how we can use higher-order models to model paths in complex networks. In this example, paths were directly given in terms of sequences of nodes traversed by some process (like a random walk). We have further seen that higher-order De Bruijn graph models can be used to capture patterns that influence the **causal topology** of a complex network, i.e. which nodes can possibly influence each other via paths. The same is true for time-respecting paths in a temporal graph. Due to the fact that time-stamped edges need to occur in the correct temporal ordering (and within a given time interval based on the maximum time difference $\delta$), the causal topology given by time-respecting paths can be very different from what we would expect from the (static) topology of links.

In the following, we will show how we can easiy and efficiently construct higher-order models for time-respecting paths in a temporal graph. To illustrate this, we use the same toy example as before:

In [ ]:
import pathpyG as pp

In [ ]:
tedges = [('a', 'b', 1),('a', 'b', 2), ('b', 'a', 3), ('b', 'c', 3), ('d', 'c', 4), ('a', 'b', 4), ('c', 'b', 4),
              ('c', 'd', 5), ('b', 'a', 5), ('c', 'b', 6)]
t = pp.TemporalGraph.from_edge_list(tedges)
print(t)

Temporal Graph with 4 nodes, 6 unique edges and 10 events in [1, 6]
{'Edge Attributes': {}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}


To better understand this temporal graph, we can again create a directed acyclic graph that represents the topology of time-respecting paths:

In [ ]:
e_i = pp.core.event_graph.EventGraph.build_edge_index(t, delta=1)
dag = pp.Graph.from_edge_index(e_i)
pp.plot(dag, node_label = [f'{v}-{w}-{time}' for v, w, time in t.temporal_edges]);

100%|██████████| 6/6 [00:00<00:00, 5608.61it/s]


<iframe srcdoc='<style>
.label-text {
 fill: #969595;
 font-size: 16px;
 font-family: sans-serif;
}
</style>

<div id = "x0ee40a5716b04f7c847d1416fe39040b"> </div>
<script charset="utf-8">
function render_plot_e5f080fd54514e0fa4817aa1001bd4c4() {
 const d3 = window.d3;
 if (!d3) { console.error('D3 not loaded'); return; }
const data = {"nodes": [{"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "0"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "1"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "2"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "3"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "4"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "5"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "6"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "7"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "8"}], "edges": [{"color": "#4c707b", "size": 2, "opacity": 0.5, "image": null, "uid": "1-2", "source": "1", "target": "2"}, {"color": "#4c707b", "size": 2, "opacity": 0.5, "image": null, "uid": "1-3", "source": "1", "target": "3"}, {"color": "#4c707b", "size": 2, "opacity": 0.5, "image": null, "uid": "2-5", "source": "2", "target": "5"}, {"color": "#4c707b", "size": 2, "opacity": 0.5, "image": null, "uid": "3-6", "source": "3", "target": "6"}, {"color": "#4c707b", "size": 2, "opacity": 0.5, "image": null, "uid": "4-7", "source": "4", "target": "7"}, {"color": "#4c707b", "size": 2, "opacity": 0.5, "image": null, "uid": "5-8", "source": "5", "target": "8"}, {"color": "#4c707b", "size": 2, "opacity": 0.5, "image": null, "uid": "6-8", "source": "6", "target": "8"}]}
const config = {"default_backend": "d3js", "cmap": "cividis", "layout": null, "width": 453.54330708661416, "height": 453.54330708661416, "latex_class_options": "", "margin": 0.1, "curvature": 0.25, "layout_window_size": [-1, -1], "delta": 1000, "separator": "->", "orientation": "down", "node": {"color": "#244a5c", "size": 15, "opacity": 0.75, "image_padding": 5}, "edge": {"color": "#4c707b", "size": 2, "opacity": 0.5}, "directed": true, "curved": true, "simulation": true, "d3js_local": false, "selector": "#x0ee40a5716b04f7c847d1416fe39040b", "show_labels": false}
console.log('static Network Template');
// Useful pages
// ------------
// https://github.com/takanori-fujiwara/d3-gallery-javascript/tree/main/animation/temporal-force-directed-graph
// https://observablehq.com/@d3/temporal-force-directed-graph
// https://observablehq.com/@mbostock/scrubber
// https://plnkr.co/edit/0VxwY1Mc5UYusvhYic8u?preview
// https://stackoverflow.com/questions/42545531/making-d3-like-force-directed-graph-with-pngs


// Create function for Network Drawing
const Network = (config) => {
 console.log("Initialize Network Function");

 // Get variables from config
 const selector = config.selector; // DOM uuid
 const width = config.width || 800; // window width 
 const height = config.height || 600; // window height
 const delta = config.delta || 300; // time between frames
 const padding = (config.node && config.node.image_padding) || 5; // distance between node and image
 const margin = config.margin || 0.1; // margin around the plot area for fixed layout
 const xlim = [-1*margin, 1+(1*margin)]; // limits of the x-coordinates
 const ylim = [-1*margin, 1+(1*margin)]; // limits of the y-coordinates
 const arrowheadMultiplier = 4; // Multiplier for arrowhead size based on edge stroke width
 const nodeStrokeWidth = 2.5; // Stroke width around nodes
 const curveFactor = config.curvature || 0.25; // Factor to control curvature of edges (higher = more curved)

 // Initialize svg canvas
 const svg = d3.select(selector)
 .append('svg')
 .attr('width', width)
 .attr('height', height)
 .attr('viewBox', [0, 0, width, height]);
 
 // add container to store network
 let container 

For $\delta=1$, we again have the following time-respecting paths:

Length one:
    a -> b
    b -> a
    b -> c
    c -> b
    c -> d
    d -> c
Length two:
    a -> b -> a (twice)
    b -> a -> b
    a -> b -> c
    b -> c -> b
    c -> b -> a
    d -> c -> d
Length three:
    a -> b -> a -> b
    b -> a -> b -> a
    a -> b -> c -> b
    b -> c -> b -> a
Length four:
    a -> b -> a -> b -> a
    a -> b -> c -> b -> a

As you can see, these time-respecting paths are actually very similar to the paths data that we have previously represented using the `PathData` object. In fact, we could - in theory - first extract all time-respecting paths of all lengths, add them to a `PathData` object and then use the `MultiOderModel` class to generate higher-order De Bruijn graph models of all orders. In the example above, since we have paths of length one to four, we could create higher-order models with orders from one to four.

However, this approach would not be efficient for large temporal graphs, as it is computationally expensive to calculate all possible time-respecting paths as well as subpaths of length $k$, especially for larger values of $\delta$. To avoid this bottleneck, `pathpyG` uses a smarter, GPU-based algorithm to calculate time-respecting paths of length $k$ that are needed for a given order $k$.

For the example above, we can generate all higher-order models up to order four as follows:

In [ ]:
m = pp.MultiOrderModel.from_temporal_graph(t, delta=1, max_order=4)

print(m.layers[3])
print(m.layers[4])

100%|██████████| 6/6 [00:00<00:00, 3136.71it/s]


Higher-order graph of order 3 with 6 nodes and 4 edges
(over 4 first-order nodes)
{   'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([4])"},
    'Graph Attributes': {'inverse_idx': "<class 'torch.Tensor'> -> torch.Size([7])", 'num_nodes': "<class 'int'>"},
    'Node Attributes': {}}
Higher-order graph of order 4 with 4 nodes and 2 edges
(over 4 first-order nodes)
{'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([2])"}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}


Remember that in a $k$-th order model nodes capture paths of length $k-1$, while edges capture paths of length $k$.

This implies that the first-order model has four nodes and six edges, which simply corresponds to the time-aggregated weighted graph for our example temporal network.

In [ ]:
print(m.layers[1])
pp.plot(m.layers[1], node_label=[v for v in m.layers[1].nodes]);

Higher-order graph of order 1 with 4 nodes and 6 edges
(over 4 first-order nodes)
{'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([6])"}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}


<iframe srcdoc='<style>
.label-text {
 fill: #969595;
 font-size: 16px;
 font-family: sans-serif;
}
</style>

<div id = "xeb4a0c5af719460c896ebd314dbbe91f"> </div>
<script charset="utf-8">
function render_plot_b2d38085a0644d60b2fe961ed4b18d89() {
 const d3 = window.d3;
 if (!d3) { console.error('D3 not loaded'); return; }
const data = {"nodes": [{"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "a"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "b"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "c"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "d"}], "edges": [{"color": "#4c707b", "size": 3.0, "opacity": 0.5, "image": null, "uid": "a-b", "source": "a", "target": "b"}, {"color": "#4c707b", "size": 2.0, "opacity": 0.5, "image": null, "uid": "b-a", "source": "b", "target": "a"}, {"color": "#4c707b", "size": 1.0, "opacity": 0.5, "image": null, "uid": "b-c", "source": "b", "target": "c"}, {"color": "#4c707b", "size": 2.0, "opacity": 0.5, "image": null, "uid": "c-b", "source": "c", "target": "b"}, {"color": "#4c707b", "size": 1.0, "opacity": 0.5, "image": null, "uid": "c-d", "source": "c", "target": "d"}, {"color": "#4c707b", "size": 1.0, "opacity": 0.5, "image": null, "uid": "d-c", "source": "d", "target": "c"}]}
const config = {"default_backend": "d3js", "cmap": "cividis", "layout": null, "width": 453.54330708661416, "height": 453.54330708661416, "latex_class_options": "", "margin": 0.1, "curvature": 0.25, "layout_window_size": [-1, -1], "delta": 1000, "separator": "->", "orientation": "down", "node": {"color": "#244a5c", "size": 15, "opacity": 0.75, "image_padding": 5}, "edge": {"color": "#4c707b", "size": 2, "opacity": 0.5}, "directed": true, "curved": true, "simulation": true, "d3js_local": false, "selector": "#xeb4a0c5af719460c896ebd314dbbe91f", "show_labels": true}
console.log('static Network Template');
// Useful pages
// ------------
// https://github.com/takanori-fujiwara/d3-gallery-javascript/tree/main/animation/temporal-force-directed-graph
// https://observablehq.com/@d3/temporal-force-directed-graph
// https://observablehq.com/@mbostock/scrubber
// https://plnkr.co/edit/0VxwY1Mc5UYusvhYic8u?preview
// https://stackoverflow.com/questions/42545531/making-d3-like-force-directed-graph-with-pngs


// Create function for Network Drawing
const Network = (config) => {
 console.log("Initialize Network Function");

 // Get variables from config
 const selector = config.selector; // DOM uuid
 const width = config.width || 800; // window width 
 const height = config.height || 600; // window height
 const delta = config.delta || 300; // time between frames
 const padding = (config.node && config.node.image_padding) || 5; // distance between node and image
 const margin = config.margin || 0.1; // margin around the plot area for fixed layout
 const xlim = [-1*margin, 1+(1*margin)]; // limits of the x-coordinates
 const ylim = [-1*margin, 1+(1*margin)]; // limits of the y-coordinates
 const arrowheadMultiplier = 4; // Multiplier for arrowhead size based on edge stroke width
 const nodeStrokeWidth = 2.5; // Stroke width around nodes
 const curveFactor = config.curvature || 0.25; // Factor to control curvature of edges (higher = more curved)

 // Initialize svg canvas
 const svg = d3.select(selector)
 .append('svg')
 .attr('width', width)
 .attr('height', height)
 .attr('viewBox', [0, 0, width, height]);
 
 // add container to store network
 let container = svg.append("g");

 // initialize link
 let link = container.append("g")
 .attr("class", "edges")
 .selectAll(".link");

 // initialize node
 let node = container.append("g")
 .attr("class", "nodes")
 .selectAll("circle.node");

 // initialize label
 let label = container.append("g")
 .attr("class", "labels")
 .selectAll(".label");

 // initialize image
 let image = container.append("g")
 .attr("class", "images")
 .selectAll(".image");

 const ticked = () => {
 // 1. Update node, l

For the second-order model, we have six nodes, which map to the six different edges (each edge trivially being a time-respecting path of length one) of the temporal graph. The six edges in the second-order model represent the six different time-respecting paths of length two (see above). Since the time-respecting path $a \rightarrow b \rightarrow a$  occurs twice at different times, we have one edge with weight two.

In [ ]:
print(m.layers[2])
print(m.layers[2].data.edge_weight)
pp.plot(m.layers[2], node_label=m.layers[2].mapping.node_ids.tolist());

Higher-order graph of order 2 with 6 nodes and 6 edges
(over 4 first-order nodes)
{   'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([6])"},
    'Graph Attributes': {'inverse_idx': "<class 'torch.Tensor'> -> torch.Size([10])", 'num_nodes': "<class 'int'>"},
    'Node Attributes': {}}
tensor([2., 1., 1., 1., 1., 1.])


<iframe srcdoc='<style>
.label-text {
 fill: #969595;
 font-size: 16px;
 font-family: sans-serif;
}
</style>

<div id = "x678706d5317f48f6aedb683b67e36cfb"> </div>
<script charset="utf-8">
function render_plot_adf9101da76947d0a9f18c8cd63f135d() {
 const d3 = window.d3;
 if (!d3) { console.error('D3 not loaded'); return; }
const data = {"nodes": [{"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "a->b"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "b->a"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "b->c"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "c->b"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "c->d"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "d->c"}], "edges": [{"color": "#4c707b", "size": 2.0, "opacity": 0.5, "image": null, "uid": "a->b-b->a", "source": "a->b", "target": "b->a"}, {"color": "#4c707b", "size": 1.0, "opacity": 0.5, "image": null, "uid": "a->b-b->c", "source": "a->b", "target": "b->c"}, {"color": "#4c707b", "size": 1.0, "opacity": 0.5, "image": null, "uid": "b->a-a->b", "source": "b->a", "target": "a->b"}, {"color": "#4c707b", "size": 1.0, "opacity": 0.5, "image": null, "uid": "b->c-c->b", "source": "b->c", "target": "c->b"}, {"color": "#4c707b", "size": 1.0, "opacity": 0.5, "image": null, "uid": "c->b-b->a", "source": "c->b", "target": "b->a"}, {"color": "#4c707b", "size": 1.0, "opacity": 0.5, "image": null, "uid": "d->c-c->d", "source": "d->c", "target": "c->d"}]}
const config = {"default_backend": "d3js", "cmap": "cividis", "layout": null, "width": 453.54330708661416, "height": 453.54330708661416, "latex_class_options": "", "margin": 0.1, "curvature": 0.25, "layout_window_size": [-1, -1], "delta": 1000, "separator": "->", "orientation": "down", "node": {"color": "#244a5c", "size": 15, "opacity": 0.75, "image_padding": 5}, "edge": {"color": "#4c707b", "size": 2, "opacity": 0.5}, "directed": true, "curved": true, "simulation": true, "d3js_local": false, "selector": "#x678706d5317f48f6aedb683b67e36cfb", "show_labels": true}
console.log('static Network Template');
// Useful pages
// ------------
// https://github.com/takanori-fujiwara/d3-gallery-javascript/tree/main/animation/temporal-force-directed-graph
// https://observablehq.com/@d3/temporal-force-directed-graph
// https://observablehq.com/@mbostock/scrubber
// https://plnkr.co/edit/0VxwY1Mc5UYusvhYic8u?preview
// https://stackoverflow.com/questions/42545531/making-d3-like-force-directed-graph-with-pngs


// Create function for Network Drawing
const Network = (config) => {
 console.log("Initialize Network Function");

 // Get variables from config
 const selector = config.selector; // DOM uuid
 const width = config.width || 800; // window width 
 const height = config.height || 600; // window height
 const delta = config.delta || 300; // time between frames
 const padding = (config.node && config.node.image_padding) || 5; // distance between node and image
 const margin = config.margin || 0.1; // margin around the plot area for fixed layout
 const xlim = [-1*margin, 1+(1*margin)]; // limits of the x-coordinates
 const ylim = [-1*margin, 1+(1*margin)]; // limits of the y-coordinates
 const arrowheadMultiplier = 4; // Multiplier for arrowhead size based on edge stroke width
 const nodeStrokeWidth = 2.5; // Stroke width around nodes
 const curveFactor = config.curvature || 0.25; // Factor to control curvature of edges (higher = more curved)

 // Initialize svg canvas
 const svg = d3.select(selector)
 .append('svg')
 .attr('width', width)
 .attr('height', height)
 .attr('viewBox', [0, 0, width, height]);
 
 // add container to store network
 let container = svg.append("g");

 // initialize link
 let link = container.append("g")
 .attr("class", "edges")
 .selectAll(".link");

 // initialize node
 let node = container.append("g")
 .attr("class", "nodes")
 .selectAll("circle.node");

 // initial

For the third-oder mode, we have four edges representing the four diffeerent time-respecting paths of length three in the temporal graph above:

In [ ]:
print(m.layers[3])
pp.plot(m.layers[3], node_label=m.layers[3].mapping.node_ids.tolist());

Higher-order graph of order 3 with 6 nodes and 4 edges
(over 4 first-order nodes)
{   'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([4])"},
    'Graph Attributes': {'inverse_idx': "<class 'torch.Tensor'> -> torch.Size([7])", 'num_nodes': "<class 'int'>"},
    'Node Attributes': {}}


<iframe srcdoc='<style>
.label-text {
 fill: #969595;
 font-size: 16px;
 font-family: sans-serif;
}
</style>

<div id = "x79aac9dd0be44542874610702310bec3"> </div>
<script charset="utf-8">
function render_plot_73b82e6e427e43018228defcddedfe15() {
 const d3 = window.d3;
 if (!d3) { console.error('D3 not loaded'); return; }
const data = {"nodes": [{"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "a->b->a"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "a->b->c"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "b->a->b"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "b->c->b"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "c->b->a"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "d->c->d"}], "edges": [{"color": "#4c707b", "size": 1.0, "opacity": 0.5, "image": null, "uid": "a->b->a-b->a->b", "source": "a->b->a", "target": "b->a->b"}, {"color": "#4c707b", "size": 1.0, "opacity": 0.5, "image": null, "uid": "a->b->c-b->c->b", "source": "a->b->c", "target": "b->c->b"}, {"color": "#4c707b", "size": 1.0, "opacity": 0.5, "image": null, "uid": "b->a->b-a->b->a", "source": "b->a->b", "target": "a->b->a"}, {"color": "#4c707b", "size": 1.0, "opacity": 0.5, "image": null, "uid": "b->c->b-c->b->a", "source": "b->c->b", "target": "c->b->a"}]}
const config = {"default_backend": "d3js", "cmap": "cividis", "layout": null, "width": 453.54330708661416, "height": 453.54330708661416, "latex_class_options": "", "margin": 0.1, "curvature": 0.25, "layout_window_size": [-1, -1], "delta": 1000, "separator": "->", "orientation": "down", "node": {"color": "#244a5c", "size": 15, "opacity": 0.75, "image_padding": 5}, "edge": {"color": "#4c707b", "size": 2, "opacity": 0.5}, "directed": true, "curved": true, "simulation": true, "d3js_local": false, "selector": "#x79aac9dd0be44542874610702310bec3", "show_labels": true}
console.log('static Network Template');
// Useful pages
// ------------
// https://github.com/takanori-fujiwara/d3-gallery-javascript/tree/main/animation/temporal-force-directed-graph
// https://observablehq.com/@d3/temporal-force-directed-graph
// https://observablehq.com/@mbostock/scrubber
// https://plnkr.co/edit/0VxwY1Mc5UYusvhYic8u?preview
// https://stackoverflow.com/questions/42545531/making-d3-like-force-directed-graph-with-pngs


// Create function for Network Drawing
const Network = (config) => {
 console.log("Initialize Network Function");

 // Get variables from config
 const selector = config.selector; // DOM uuid
 const width = config.width || 800; // window width 
 const height = config.height || 600; // window height
 const delta = config.delta || 300; // time between frames
 const padding = (config.node && config.node.image_padding) || 5; // distance between node and image
 const margin = config.margin || 0.1; // margin around the plot area for fixed layout
 const xlim = [-1*margin, 1+(1*margin)]; // limits of the x-coordinates
 const ylim = [-1*margin, 1+(1*margin)]; // limits of the y-coordinates
 const arrowheadMultiplier = 4; // Multiplier for arrowhead size based on edge stroke width
 const nodeStrokeWidth = 2.5; // Stroke width around nodes
 const curveFactor = config.curvature || 0.25; // Factor to control curvature of edges (higher = more curved)

 // Initialize svg canvas
 const svg = d3.select(selector)
 .append('svg')
 .attr('width', width)
 .attr('height', height)
 .attr('viewBox', [0, 0, width, height]);
 
 // add container to store network
 let container = svg.append("g");

 // initialize link
 let link = container.append("g")
 .attr("class", "edges")
 .selectAll(".link");

 // initialize node
 let node = container.append("g")
 .attr("class", "nodes")
 .selectAll("circle.node");

 // initialize label
 let label = container.append("g")
 .attr("class", "labels")
 .selectAll(".label");

 // initialize image
 let image = container.append("g")
 .attr("class", "images")
 

And finally, for the model with order $k=4$ we only have two edges, representing the two time-respecting paths $a \rightarrow b \rightarrow a \rightarrow b \rightarrow a$ and $a \rightarrow b \rightarrow c \rightarrow b \rightarrow a$:

In [ ]:
print(m.layers[4])
pp.plot(m.layers[4], node_label=m.layers[4].mapping.node_ids.tolist());

Higher-order graph of order 4 with 4 nodes and 2 edges
(over 4 first-order nodes)
{'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([2])"}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}


<iframe srcdoc='<style>
.label-text {
 fill: #969595;
 font-size: 16px;
 font-family: sans-serif;
}
</style>

<div id = "x9886715d01044fdaaf131f32f9dda1fe"> </div>
<script charset="utf-8">
function render_plot_a1bdd8992b6c45c98397113121d90616() {
 const d3 = window.d3;
 if (!d3) { console.error('D3 not loaded'); return; }
const data = {"nodes": [{"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "a->b->a->b"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "a->b->c->b"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "b->a->b->a"}, {"color": "#244a5c", "size": 15, "opacity": 0.75, "image": null, "uid": "b->c->b->a"}], "edges": [{"color": "#4c707b", "size": 1.0, "opacity": 0.5, "image": null, "uid": "a->b->a->b-b->a->b->a", "source": "a->b->a->b", "target": "b->a->b->a"}, {"color": "#4c707b", "size": 1.0, "opacity": 0.5, "image": null, "uid": "a->b->c->b-b->c->b->a", "source": "a->b->c->b", "target": "b->c->b->a"}]}
const config = {"default_backend": "d3js", "cmap": "cividis", "layout": null, "width": 453.54330708661416, "height": 453.54330708661416, "latex_class_options": "", "margin": 0.1, "curvature": 0.25, "layout_window_size": [-1, -1], "delta": 1000, "separator": "->", "orientation": "down", "node": {"color": "#244a5c", "size": 15, "opacity": 0.75, "image_padding": 5}, "edge": {"color": "#4c707b", "size": 2, "opacity": 0.5}, "directed": true, "curved": true, "simulation": true, "d3js_local": false, "selector": "#x9886715d01044fdaaf131f32f9dda1fe", "show_labels": true}
console.log('static Network Template');
// Useful pages
// ------------
// https://github.com/takanori-fujiwara/d3-gallery-javascript/tree/main/animation/temporal-force-directed-graph
// https://observablehq.com/@d3/temporal-force-directed-graph
// https://observablehq.com/@mbostock/scrubber
// https://plnkr.co/edit/0VxwY1Mc5UYusvhYic8u?preview
// https://stackoverflow.com/questions/42545531/making-d3-like-force-directed-graph-with-pngs


// Create function for Network Drawing
const Network = (config) => {
 console.log("Initialize Network Function");

 // Get variables from config
 const selector = config.selector; // DOM uuid
 const width = config.width || 800; // window width 
 const height = config.height || 600; // window height
 const delta = config.delta || 300; // time between frames
 const padding = (config.node && config.node.image_padding) || 5; // distance between node and image
 const margin = config.margin || 0.1; // margin around the plot area for fixed layout
 const xlim = [-1*margin, 1+(1*margin)]; // limits of the x-coordinates
 const ylim = [-1*margin, 1+(1*margin)]; // limits of the y-coordinates
 const arrowheadMultiplier = 4; // Multiplier for arrowhead size based on edge stroke width
 const nodeStrokeWidth = 2.5; // Stroke width around nodes
 const curveFactor = config.curvature || 0.25; // Factor to control curvature of edges (higher = more curved)

 // Initialize svg canvas
 const svg = d3.select(selector)
 .append('svg')
 .attr('width', width)
 .attr('height', height)
 .attr('viewBox', [0, 0, width, height]);
 
 // add container to store network
 let container = svg.append("g");

 // initialize link
 let link = container.append("g")
 .attr("class", "edges")
 .selectAll(".link");

 // initialize node
 let node = container.append("g")
 .attr("class", "nodes")
 .selectAll("circle.node");

 // initialize label
 let label = container.append("g")
 .attr("class", "labels")
 .selectAll(".label");

 // initialize image
 let image = container.append("g")
 .attr("class", "images")
 .selectAll(".image");

 const ticked = () => {
 // 1. Update node, label, and image positions
 node.attr("transform", d => `translate(${d.x},${d.y})`);
 label.attr("transform", d => `translate(${d.x},${d.y})`);
 image.attr("transform", d => `translate(${d.x},${d.y})`);

 // 2. Update the link paths
 link.attr('d', d => {
 // --- A. Get necessary properties ---
 const sourceRadius = d.source.size |

Intuitively, since in our example there are no time-respecting paths longer than four, if we were to generate a multi-order model with De Bruijn graphs with orders larger than four, those graphs cannot contain any edges. We see this in the following example. The first-order graph is simply the time-aggregated weighted graph, i.e. the number of nodes is equal to the number of nodes in the temporal graph and the number of edges is equal to the number of different time-stamped edges. In each graph of order $k>1$, the number of nodes corresponds to the number of edges in the graph with order $k-1$, since each of those nodes corresponds to a time-respecting path of length $k-1$, which are represented by edges in a $k-1$-th order gaph. This implies that the graph with order five has two nodes, which are the two time-respecting paths of length four. Those nodes are not connected since there is no time-respecting path with length five.

In [ ]:
m_1 = pp.MultiOrderModel.from_temporal_graph(t, delta=1, max_order=5)
print(m_1.layers[4])
print(m_1.layers[5])
pp.plot(m_1.layers[5], node_label=m_1.layers[5].mapping.node_ids.tolist())

100%|██████████| 6/6 [00:00<00:00, 8300.07it/s]


Higher-order graph of order 4 with 4 nodes and 2 edges
(over 4 first-order nodes)
{'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([2])"}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}
Higher-order graph of order 5 with 2 nodes and 0 edges
(over 4 first-order nodes)
{'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([0])"}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}


<pathpyG.visualisations._d3js.backend.D3jsBackend object at 0x7a27eec7c830>

## Constructing Higher-Order De Bruijn Graph Models for Empirical Temporal Networks

Let us now use `pathpyG` to construcct higher-order De Bruijn graph models for time-respecting paths in empirical temporal network. For this, we first read a number of temporal graphs using `TemporalGraph.from_csv`. In the following, we use the following three publicly available data sets:

- Antenna interactions between ants in a colony [(Blonder and Dornhaus, 2011)](https://pubmed.ncbi.nlm.nih.gov/21625450/)
- E-Mail exchanges in a manufacturing company [(Nurek and Michalski, 2020)](https://www.ii.pwr.edu.pl/~michalski/index.php?content=datasets)
- Face-to-face interactions in a highschool [(Mastrandrea, Fournet, Barrat, 2015)](http://www.sociopatterns.org/datasets/high-school-contact-and-friendship-networks/)

In [ ]:
t_ants = pp.io.read_csv_temporal_graph(pp.io.example_data('ants_1_1.tedges'), header=False)
print(t_ants)

Temporal Graph with 89 nodes, 947 unique edges and 1911 events in [0, 1438]
{   'Edge Attributes': {'edge_attr_1': "<class 'torch.Tensor'> -> torch.Size([1911])", 'edge_attr_2': "<class 'torch.Tensor'> -> torch.Size([1911])"},
    'Graph Attributes': {'num_nodes': "<class 'int'>"},
    'Node Attributes': {}}


In [ ]:
t_email = pp.io.read_csv_temporal_graph(pp.io.example_data('manufacturing_email.tedges'), header=False)
print(t_email)

Temporal Graph with 167 nodes, 5784 unique edges and 82614 events in [0, 23430482000000]
{'Edge Attributes': {}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}


In [ ]:
t_sp = pp.io.read_csv_temporal_graph(pp.io.example_data('sociopatterns_highschool_2013.tedges'), header=False)
print(t_sp)

Temporal Graph with 327 nodes, 5818 unique edges and 188508 events in [1385982020, 1386345580]
{   'Edge Attributes': {'edge_attr_1': "<class 'numpy.ndarray'>", 'edge_attr_2': "<class 'numpy.ndarray'>"},
    'Graph Attributes': {'num_nodes': "<class 'int'>"},
    'Node Attributes': {}}


To generate a `MultiOderModel` consisting of multiple layers of higher-order De Bruijn graph models, we can use the `MultiOderModel.from_temporal_graph` method. We can further specify the maximum order of the highest-order layer, as well as the maximum time difference $\delta$ for time-respecting paths.

For the ants, we consider time-respecting paths with a maximum time difference of 30 seconds up to length four:

In [ ]:
m_2 = pp.MultiOrderModel.from_temporal_graph(t_ants, delta=30, max_order=4)
print(m_2.layers[1])
print(m_2.layers[2])
print(m_2.layers[3])
print(m_2.layers[4])

100%|██████████| 883/883 [00:00<00:00, 8992.98it/s]


Higher-order graph of order 1 with 89 nodes and 947 edges
(over 89 first-order nodes)
{'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([947])"}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}
Higher-order graph of order 2 with 947 nodes and 1780 edges
(over 89 first-order nodes)
{   'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([1780])"},
    'Graph Attributes': {'inverse_idx': "<class 'torch.Tensor'> -> torch.Size([1911])", 'num_nodes': "<class 'int'>"},
    'Node Attributes': {}}
Higher-order graph of order 3 with 1780 nodes and 2410 edges
(over 89 first-order nodes)
{   'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([2410])"},
    'Graph Attributes': {'inverse_idx': "<class 'torch.Tensor'> -> torch.Size([2518])", 'num_nodes': "<class 'int'>"},
    'Node Attributes': {}}
Higher-order graph of order 4 with 2410 nodes and 3292 edges
(over 89 first-order nodes)
{   'Edge Attributes'

For the E-Mail communication network, we use time-respecting paths up to length four with a maximum time difference of one hour. Note that the timestamps in this data set are given in microseconds rather than seconds, so we scale $\delta$ accordingly.

In [ ]:
m_3 = pp.MultiOrderModel.from_temporal_graph(t_email, delta=3600 * 1000000, max_order=4)
print(m_3.layers[1])
print(m_3.layers[2])
print(m_3.layers[3])
print(m_3.layers[4])

100%|██████████| 57842/57842 [00:16<00:00, 3581.34it/s]


Higher-order graph of order 1 with 167 nodes and 5784 edges
(over 167 first-order nodes)
{'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([5784])"}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}
Higher-order graph of order 2 with 5784 nodes and 26080 edges
(over 167 first-order nodes)
{   'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([26080])"},
    'Graph Attributes': {'inverse_idx': "<class 'torch.Tensor'> -> torch.Size([82614])", 'num_nodes': "<class 'int'>"},
    'Node Attributes': {}}
Higher-order graph of order 3 with 26080 nodes and 50861 edges
(over 167 first-order nodes)
{   'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([50861])"},
    'Graph Attributes': {'inverse_idx': "<class 'torch.Tensor'> -> torch.Size([115555])", 'num_nodes': "<class 'int'>"},
    'Node Attributes': {}}
Higher-order graph of order 4 with 50861 nodes and 77916 edges
(over 167 first-order nodes)
{  

And finally, for the largest data set from the Sociopatterns collaboration, we use a maximum time difference of 15 minutes. As you can see below, we can efficiently generate a 5-th order model despite using a temporal graph with more than 188,000 time-stamped edges and considering all time-respecting paths up to length five with a large maximum time difference. Thanks to the use of GPU-accelerated operations, creating such a model takes less than 12 seconds on an (old) RTX 2080 GPU.

In [ ]:
m_4 = pp.MultiOrderModel.from_temporal_graph(t_sp, delta=900, max_order=5)
print(m_4.layers[1])
print(m_4.layers[5])

100%|██████████| 7375/7375 [00:04<00:00, 1558.36it/s]


Higher-order graph of order 1 with 327 nodes and 5818 edges
(over 327 first-order nodes)
{'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([5818])"}, 'Graph Attributes': {'num_nodes': "<class 'int'>"}, 'Node Attributes': {}}
Higher-order graph of order 5 with 24550 nodes and 17433 edges
(over 327 first-order nodes)
{   'Edge Attributes': {'edge_weight': "<class 'torch.Tensor'> -> torch.Size([17433])"},
    'Graph Attributes': {'inverse_idx': "<class 'torch.Tensor'> -> torch.Size([12314285])", 'num_nodes': "<class 'int'>"},
    'Node Attributes': {}}


How can we use such higher-order graph models? Let us reconsider our example temporal network with temporal communities from notebook 02-02.